# Combined TI × LI × SI Pipeline (aligned with final_wingx + wingx-smid-kmeans)

**Merges Time-of-Day Index (TI), Location Index (LI), and Seasonality Index (SI) with raw indices and multipliers**

**Input**: `lj_smid_us_revenue_clean_24_25_metro_local_TimeZone.parquet`  
**Output**: `combined_ti_li_si_output.csv`

**Final schema**: Full audit trail with raw indices AND multipliers per CTS framework

**Scope**: 16 priority corridors (6 Light Jet + 10 Super Midsize Jet) × 12 months × 7 DOW × 6 TOD = **8,064 rows**

**Key alignment**: Hours filters (LJ ≥1.5, SMID ≥2.5) matching final_wingx and wingx-smid-kmeans notebooks

## Configuration

In [ ]:
import pandas as pd
import numpy as np

# Input file (local)
RAW_PARQUET = 'wheelsup_processed_wingx_lj_smid_us_revenue_clean_24_25_metro_local_TimeZone.parquet'

# Time periods
SELECTED_YEARS = [2024, 2025]
MEAN_CELL_SHARE = 1 / 42  # 6 TOD × 7 DOW = 42 cells per corridor

# Filtering thresholds (aligned with source notebooks)
MIN_MISSIONS_FOR_SI = 30  # Minimum annual missions per corridor before computing SI
MIN_HOURS_LJ = 1.5        # Hours filter for Light Jet (final_wingx Cell 4)
MIN_HOURS_SMID = 2.5      # Hours filter for SMID (wingx-smid-kmeans Cell 2)

# Aircraft models (fixed: now 16 models matching final_wingx)
LIGHT_JET_MODELS = [
    'Phenom 300', 'Phenom 300-E',
    'Citation CJ3', 'Citation CJ3+', 'Citation CJ4', 'Citation CJ4 Gen2',
    'Citation CJ2', 'Citation CJ2+',                           # CJ2 variants added
    'Hawker 400XP', '400-A', '400-XP', 'Beechjet 400',        # 400-XP, Beechjet 400 added
    'Citation Ultra', 'Citation V',
    'Citation Encore', 'Citation Encore+'                      # Citation Encore (non-plus) added
]

WU_OPERATORS = [
    'Wheels Up Private Jets', 'Wheels Up LLC',
    'Mountain Aviation', 'Alante Air Charter'
]

# Time categories
DAY_ORDER    = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
BUCKET_ORDER = ['00:00-06:59', '07:00-08:59', '09:00-12:59',
                '13:00-15:59', '16:00-18:59', '19:00-23:59']
MONTH_NAMES  = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

# TI Score mapping (label → score)
TI_SCORE_MAP = {
    "> 3.0x":             10,
    "> 2.0x to 3.0x":     10,
    "> 1.75x to 2.0x":     9,
    "> 1.5x to 1.75x":     8,
    "> 1.25x to 1.5x":     7,
    "> 1.1x to 1.25x":     6,
    "> 1.0x to 1.1x":      5,
    "> 0.75x to 1.0x":     4,
    "> 0.6x to 0.75x":     3,
    "> 0.5x to 0.6x":      2,
    "> 0 to <= 0.5x":      1,
}
TI_BINS   = [0, 0.5, 0.6, 0.75, 1.0, 1.1, 1.25, 1.5, 1.75, 2.0, 3.0, float('inf')]
TI_LABELS = list(TI_SCORE_MAP.keys())

# SI Multiplier mapping (label → multiplier)
SI_MULTIPLIER_MAP = {
    "< 0.75x":            0.75,
    "0.75x to < 0.90x":   0.90,
    "0.90x to < 1.10x":   1.00,
    "1.10x to < 1.30x":   1.15,
    "1.30x to < 1.60x":   1.30,
    ">= 1.60x":           1.50,
}
SI_BINS   = [0, 0.75, 0.90, 1.10, 1.30, 1.60, float('inf')]
SI_LABELS = list(SI_MULTIPLIER_MAP.keys())

# LI Multiplier mapping (label → multiplier)
LI_MULTIPLIER_MAP = {
    "<= 0.75x":           0.85,
    "> 0.75x to 1.00x":   0.95,
    "> 1.00x to 1.50x":   1.00,
    "> 1.50x to 2.50x":   1.10,
    "> 2.50x to 4.00x":   1.20,
    "> 4.00x":            1.30,
}
LI_BINS   = [0, 0.75, 1.00, 1.50, 2.50, 4.00, float('inf')]
LI_LABELS = list(LI_MULTIPLIER_MAP.keys())

# Priority corridors
LJ_CORRIDORS = [
    'Denver->LA Basin',           'LA Basin->Denver',
    'Charlotte->South Florida',   'South Florida->Charlotte',
    'Chicago->South Florida',     'South Florida->Chicago',
]

SMID_CORRIDORS = [
    'New York City->South Florida', 'South Florida->New York City',
    'Denver->South Florida',        'South Florida->Denver',
    'Chicago->South Florida',       'South Florida->Chicago',
    'Phoenix Valley->Boston',       'Boston->Phoenix Valley',
    'LA Basin->New York City',      'New York City->LA Basin',
]

print("✓ Configuration loaded")
print(f"  LJ corridors: {len(LJ_CORRIDORS)} (hours filter: >= {MIN_HOURS_LJ})")
print(f"  SMID corridors: {len(SMID_CORRIDORS)} (hours filter: >= {MIN_HOURS_SMID})")
print(f"  LJ models: {len(LIGHT_JET_MODELS)} (was 12, now 16 - matching final_wingx)")
print(f"  Min SI missions: {MIN_MISSIONS_FOR_SI}")

## Step 1: Load & Preprocess

In [ ]:
df = pd.read_parquet(RAW_PARQUET)
df['FlightDate_local'] = pd.to_datetime(df['FlightDate_local'])
df['year_local']  = df['FlightDate_local'].dt.year
df['month_local'] = df['FlightDate_local'].dt.month
df = df.rename(columns={'FromCluster': 'FromMetro', 'ToCluster': 'ToMetro'})

print(f"✓ Loaded {len(df):,} records")
print(f"  Date range: {df['FlightDate_local'].min().date()} to {df['FlightDate_local'].max().date()}")

# Filter to selected years and inter-metro
data = df[
    df['year_local'].isin(SELECTED_YEARS) &
    (df['FromMetro'] != df['ToMetro'])
].copy()

# Build corridors (format: "City->City" no spaces)
data['Route']   = data['FromMetro'] + '->' + data['ToMetro']
data['is_wu']   = data['Operator'].isin(WU_OPERATORS)

# Categorize days and time buckets
data['day_name']    = pd.Categorical(data['dow_local'],         categories=DAY_ORDER,    ordered=True)
data['hour_bucket'] = pd.Categorical(data['hour_bucket_local'], categories=BUCKET_ORDER, ordered=True)

print(f"✓ Preprocessed {len(data):,} inter-metro records")

## Step 2: Segment Split

In [ ]:
smid_data = data[data['aircraft_segment'] == 'Super Midsize Jet'].copy()
lj_data   = data[data['aircraft_model'].isin(LIGHT_JET_MODELS)].copy()

print(f"✓ LJ records (all): {len(lj_data):,}")
print(f"  LJ models in data: {lj_data['aircraft_model'].nunique()}")
print(f"✓ SMID records (all): {len(smid_data):,}")

## Step 3: SI Computation (with hours filter)

In [ ]:
def compute_si(df, corridors, cabin_label, min_hours):
    """
    Compute Seasonality Index multiplier per corridor × month.
    ALIGNED WITH WINGX LOGIC:
    1. Filter by min_hours (1.5 for LJ, 2.5 for SMID)
    2. Group by corridor × month
    3. Filter to corridors with > MIN_MISSIONS annual missions
    4. Compute monthly_density = monthly_flights / annual_flights
    5. Compute SI = monthly_density / (1/12)
    6. Bin to SI_multiplier
    
    Returns: corridor, cabin, month, month_num, monthly_density, seasonality_index, si_slab, SI_multiplier
    """
    # Filter by hours AND corridor list
    df_corr = df[(df['Route'].isin(corridors)) & (df['Hours'] >= min_hours)].copy()
    
    # Step 1: Monthly flight counts (wingx style)
    monthly = df_corr.groupby(['Route','month_local']).size().unstack(fill_value=0)
    monthly = monthly.reindex(columns=range(1,13), fill_value=0)
    
    # Step 2: FILTER - wingx logic - corridors with > MIN_MISSIONS annual missions
    annual = monthly.sum(axis=1)
    mask_min_missions = annual > MIN_MISSIONS_FOR_SI
    monthly_filtered = monthly[mask_min_missions].copy()
    annual_filtered = annual[mask_min_missions].copy()
    
    if len(monthly_filtered) == 0:
        print(f"  ⚠️  [{cabin_label}] No corridors passed filters (hours >= {min_hours} AND missions > {MIN_MISSIONS_FOR_SI})")
        return pd.DataFrame()
    
    print(f"  ✓ [{cabin_label}] {len(monthly_filtered)} corridors passed both filters (hours >= {min_hours}, missions > {MIN_MISSIONS_FOR_SI})")
    
    # Step 3: Normalize to monthly density (wingx style)
    density = monthly_filtered.div(annual_filtered, axis=0)
    
    # Step 4: Create seasonality index (wingx style)
    si = density / (1/12)
    
    # Convert to long format
    si_long = si.stack().rename('seasonality_index').reset_index()
    si_long.columns = ['corridor','month_num','seasonality_index']
    
    # Add monthly_density
    density_long = density.stack().rename('monthly_density').reset_index()
    density_long.columns = ['corridor','month_num','monthly_density']
    si_long = si_long.merge(density_long, on=['corridor','month_num'])
    
    # Bin into SI slab label
    si_long['si_slab'] = pd.cut(
        si_long['seasonality_index'],
        bins=SI_BINS,
        labels=SI_LABELS,
        right=False
    )
    
    # Map slab to multiplier
    si_long['SI_multiplier'] = si_long['si_slab'].map(SI_MULTIPLIER_MAP).astype(float)
    
    # Add month name
    si_long['month'] = si_long['month_num'].map(MONTH_NAMES)
    si_long['cabin'] = cabin_label
    
    return si_long[['corridor','cabin','month','month_num','monthly_density','seasonality_index','si_slab','SI_multiplier']]

print("\n🔄 Computing SI (with hours filters):")
lj_si   = compute_si(lj_data,   LJ_CORRIDORS,   'Light Jet', MIN_HOURS_LJ)
smid_si = compute_si(smid_data, SMID_CORRIDORS, 'Super Midsize Jet', MIN_HOURS_SMID)

print(f"\n✓ LJ SI: {len(lj_si)} rows (filtered)")
print(f"✓ SMID SI: {len(smid_si)} rows (filtered)")

## Step 4: TI Computation (with hours filter)

In [ ]:
def compute_ti(df, corridors, cabin_label, min_hours):
    """
    Compute TI Score (0-10) per corridor × day × hour_bucket.
    Includes hours filter before groupby.
    Returns: corridor, cabin, DOW, TOD, cell_share, ratio_vs_mean, ti_slab, TI_score
    """
    df_corr = df[(df['Route'].isin(corridors)) & (df['Hours'] >= min_hours)].copy()
    
    # Count flights in each (Route, day, time) cell
    grouped = (
        df_corr.groupby(['Route','day_name','hour_bucket'], observed=False)
        .size()
        .reset_index(name='flight_count')
    )
    
    # Total flights per corridor (across all cells)
    grouped['total_corridor_flights'] = grouped.groupby('Route')['flight_count'].transform('sum')
    
    # Cell share (what fraction of corridor flights are in this cell)
    grouped['cell_share'] = grouped['flight_count'] / grouped['total_corridor_flights']
    
    # Ratio vs mean (how much above/below flat 1/42)
    grouped['ratio_vs_mean'] = grouped['cell_share'] / MEAN_CELL_SHARE
    
    # Bin into TI slab label
    grouped['ti_slab'] = pd.cut(
        grouped['ratio_vs_mean'],
        bins=TI_BINS,
        labels=TI_LABELS,
        right=True,
        ordered=False
    )
    
    # Map slab to TI score
    grouped['TI_score'] = grouped['ti_slab'].map(TI_SCORE_MAP).astype('Int64')
    
    # Mark zero-flight cells
    grouped.loc[grouped['flight_count'] == 0, ['ti_slab', 'TI_score']] = [None, 0]
    
    # Rename for final output
    grouped = grouped.rename(columns={'Route':'corridor','day_name':'DOW','hour_bucket':'TOD'})
    grouped['cabin'] = cabin_label
    
    return grouped[['corridor','cabin','DOW','TOD','cell_share','ratio_vs_mean','ti_slab','TI_score']]

lj_ti   = compute_ti(lj_data,   LJ_CORRIDORS,   'Light Jet', MIN_HOURS_LJ)
smid_ti = compute_ti(smid_data, SMID_CORRIDORS, 'Super Midsize Jet', MIN_HOURS_SMID)

print(f"✓ LJ TI: {len(lj_ti)} rows")
print(f"✓ SMID TI: {len(smid_ti)} rows")

## Step 5: LI Computation (with hours filter)

In [ ]:
def compute_li(df, corridors, cabin_label, min_hours):
    """
    Compute Location Index multiplier per corridor.
    LI_index = (corridor WU share) / (network WU share)
    Includes hours filter before computing shares.
    Returns: corridor, cabin, corridor_wu_share, network_wu_share, LI_index, li_slab, LI_multiplier
    """
    # Filter by hours first
    df_hours_filtered = df[df['Hours'] >= min_hours].copy()
    
    # Network-wide WU operator share (after hours filter)
    network_wu_share = df_hours_filtered['is_wu'].sum() / len(df_hours_filtered)
    
    # Filter to priority corridors
    df_corr = df_hours_filtered[df_hours_filtered['Route'].isin(corridors)]
    
    # Corridor-level aggregation
    agg = (
        df_corr.groupby('Route')
        .agg(
            corridor_total=('is_wu', 'count'),
            corridor_wu=('is_wu', 'sum')
        )
        .reset_index()
    )
    
    # WU share on corridor
    agg['corridor_wu_share'] = agg['corridor_wu'] / agg['corridor_total']
    
    # LI Index
    agg['LI_index'] = agg['corridor_wu_share'] / network_wu_share
    
    # Bin into LI slab label
    agg['li_slab'] = pd.cut(
        agg['LI_index'],
        bins=LI_BINS,
        labels=LI_LABELS,
        right=True,
        include_lowest=True,
        ordered=False
    )
    
    # Map slab to LI multiplier
    agg['LI_multiplier'] = agg['li_slab'].map(LI_MULTIPLIER_MAP).astype(float)
    
    agg = agg.rename(columns={'Route':'corridor'})
    agg['cabin'] = cabin_label
    agg['network_wu_share'] = network_wu_share
    
    print(f"[{cabin_label}] Network WU share (hours >= {min_hours}): {network_wu_share:.4%}")
    
    return agg[['corridor','cabin','corridor_wu_share','network_wu_share','LI_index','li_slab','LI_multiplier']]

lj_li   = compute_li(lj_data,   LJ_CORRIDORS,   'Light Jet', MIN_HOURS_LJ)
smid_li = compute_li(smid_data, SMID_CORRIDORS, 'Super Midsize Jet', MIN_HOURS_SMID)

print(f"✓ LJ LI: {len(lj_li)} rows")
print(f"✓ SMID LI: {len(smid_li)} rows")

## Step 6: Combine All Data

In [ ]:
# Concatenate by index type
si_all = pd.concat([lj_si, smid_si], ignore_index=True)
ti_all = pd.concat([lj_ti, smid_ti], ignore_index=True)
li_all = pd.concat([lj_li, smid_li], ignore_index=True)

print(f"✓ SI combined: {len(si_all)} rows")
print(f"✓ TI combined: {len(ti_all)} rows")
print(f"✓ LI combined: {len(li_all)} rows")

## Step 7: Build Final Table

In [ ]:
# Cross TI grid with 12 months
months_df = pd.DataFrame({'month_num': range(1,13)})
months_df['month'] = months_df['month_num'].map(MONTH_NAMES)

# Start with TI (corridor × day × time) and cross with months
final = ti_all.merge(months_df, how='cross')

print(f"✓ After cross-join with months: {len(final):,} rows")

# Join SI (by corridor × cabin × month) — only for corridors that passed MIN_MISSIONS filter
final = final.merge(
    si_all[['corridor','cabin','month','monthly_density','seasonality_index','si_slab','SI_multiplier']],
    on=['corridor','cabin','month'],
    how='left'
)

print(f"✓ After SI join: {len(final):,} rows")
print(f"  SI nulls (corridors filtered out by MIN_MISSIONS): {final['SI_multiplier'].isnull().sum():,}")

# Join LI (by corridor × cabin)
final = final.merge(
    li_all[['corridor','cabin','corridor_wu_share','network_wu_share','LI_index','li_slab','LI_multiplier']],
    on=['corridor','cabin'],
    how='left'
)

print(f"✓ After LI join: {len(final):,} rows")

## Step 8: Finalize Output

In [ ]:
# Rename cabin → segment
final = final.rename(columns={'cabin':'segment', 'month':'Month'})

# Select and reorder final columns (18 total)
final = final[[
    'corridor', 'segment', 'TOD', 'DOW', 'Month',
    'cell_share', 'ratio_vs_mean', 'ti_slab', 'TI_score',
    'monthly_density', 'seasonality_index', 'si_slab', 'SI_multiplier',
    'corridor_wu_share', 'network_wu_share', 'LI_index', 'li_slab', 'LI_multiplier'
]]

# Sort for readability
final = final.sort_values(
    ['segment', 'corridor', 'Month', 'DOW', 'TOD']
).reset_index(drop=True)

# Validation
print(f"\n✅ FINAL TABLE COMPLETE")
print(f"\n📊 Summary:")
print(f"  Rows: {len(final):,}")
print(f"  Columns: {len(final.columns)}")
print(f"  Unique corridors: {final['corridor'].nunique()}")
print(f"  Unique segments: {final['segment'].nunique()}")
print(f"  Unique months: {final['Month'].nunique()}")

print(f"\n🔍 Hours filter impact (SI):")
si_pass_rate = (final['SI_multiplier'].notna().sum() / len(final)) * 100
print(f"  SI_multiplier not null: {final['SI_multiplier'].notna().sum():,} rows ({si_pass_rate:.1f}%)")
print(f"  SI_multiplier nulls (hours filter removed): {final['SI_multiplier'].isnull().sum():,} rows")

print(f"\n🔍 Other null checks:")
print(f"  cell_share nulls: {final['cell_share'].isnull().sum()}")
print(f"  ti_slab nulls: {final['ti_slab'].isnull().sum()}")
print(f"  LI_multiplier nulls: {final['LI_multiplier'].isnull().sum()}")

print(f"\n📋 Segment breakdown:")
for segment in final['segment'].unique():
    count = len(final[final['segment'] == segment])
    si_null = final[(final['segment']==segment) & (final['SI_multiplier'].isnull())].shape[0]
    print(f"  {segment}: {count:,} rows (SI nulls from hours filter: {si_null:,})")

print(f"\n🔢 Index distribution:")
print(f"  ratio_vs_mean: {final['ratio_vs_mean'].min():.4f} to {final['ratio_vs_mean'].max():.4f}")
print(f"  seasonality_index: {final['seasonality_index'].min():.4f} to {final['seasonality_index'].max():.4f}")
print(f"  LI_index: {final['LI_index'].min():.4f} to {final['LI_index'].max():.4f}")

print(f"\n📍 Sample rows (with SI data):")
display(final[final['SI_multiplier'].notna()].head(20))

## Step 9: Export

In [ ]:
output_path = "combined_ti_li_si_output.csv"
final.to_csv(output_path, index=False)

print(f"\n✅ EXPORTED: {output_path}")
print(f"\n📋 Verification:")
check = pd.read_csv(output_path)
print(f"  Rows: {len(check):,}")
print(f"  Columns: {len(check.columns)}")
print(f"  SI data rows: {check['SI_multiplier'].notna().sum():,}")
print(f"\n✓ Pipeline complete (final_wingx + wingx-smid-kmeans aligned)")
print(f"  LJ hours filter: >= {MIN_HOURS_LJ}")
print(f"  SMID hours filter: >= {MIN_HOURS_SMID}")
print(f"  LJ models: {len(LIGHT_JET_MODELS)} (corrected from 12 to 16)")